# Reproduction: Saini et al. (2023) hybrid RF+XGBoost

Paper's methods only: 5 baselines (DT, LR, RF, XGBoost, MLP) plus the paper's RF+XGBoost averaging hybrid, with SHAP.

Runs every dataset in `DATASETS` in one go and ends with one combined table. Remove datasets from the list to run fewer.

Setup on Kaggle: turn Internet ON, add the saved `kaggle_prep` notebook output as an input (or the raw-data dataset, which is slower), run all cells. Results are written to `/kaggle/working/results/<dataset>/`.

In [ ]:
DATASETS = ["nsl_kdd", "unsw_nb15", "unsw_nb15_nodedup", "cse_cic_ids2018", "cic_ids2017"]
MODELS = ""   # "" = all; or a subset, e.g. "averaging,random forest"
MODE = "reproduce"

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

# Clone (or update) the repo; needs Internet turned on in notebook settings
REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"

# The uploaded raw folder (containing nsl_kdd/, unsw_nb15/, cic-ids-2017/, cds-ids-2018/) is read
# in place from /kaggle/input: nothing is copied, so saved versions stay small.
RAW_MARKERS = {
    "nsl_kdd": "nsl_kdd/KDDTrain+*",
    "unsw_nb15": "unsw_nb15/UNSW_NB15_training-set.csv",
    "unsw_nb15_nodedup": "unsw_nb15/UNSW_NB15_training-set.csv",
    "cic_ids2017": "cic*2017/*WorkingHours*.csv",
    "cse_cic_ids2018": "c*2018/*-2018.csv",
}

def point_at_raw(datasets):
    roots, missing = set(), []
    for ds in datasets:
        hits = glob.glob(f"/kaggle/input/**/{RAW_MARKERS[ds]}", recursive=True)
        if hits:
            roots |= {os.path.dirname(os.path.dirname(h)) for h in hits}
        else:
            missing.append(ds)
    if missing or len(roots) != 1:
        for root, _, files in os.walk("/kaggle/input"):
            for fn in files:
                print("  ", os.path.join(root, fn))
        raise AssertionError(f"raw data not found for {missing}, or found in several places: {roots}")
    os.environ["RIS_DATA_RAW"] = roots.pop()
    print("raw data folder:", os.environ["RIS_DATA_RAW"])

def stage(ds):
    """Use prepared data (kaggle_prep output) if attached; otherwise point at the raw folder.
    Returns True when cleaning can be skipped."""
    prepared = glob.glob(f"/kaggle/input/**/{ds}/dataset_final.csv", recursive=True)
    if prepared:
        pdest = f"{WORK}/data/processed/{ds}"
        os.makedirs(pdest, exist_ok=True)
        shutil.copy(prepared[0], pdest)  # small file; the pipeline writes next to it
        print(f"{ds}: using prepared data {prepared[0]}")
        return True
    point_at_raw([ds])
    return False

In [ ]:
!pip install -q imbalanced-learn shap xgboost

In [ ]:
import time
status = {}
for ds in DATASETS:
    print("=" * 30, ds, "=" * 30, flush=True)
    t0 = time.time()
    skip = stage(ds)
    proc = subprocess.run(
        [sys.executable, "-u", "run_pipeline.py", "--dataset", ds, "--mode", MODE] + (["--skip-prep"] if skip else []) + (["--models", MODELS] if MODELS else []),
        cwd=SRC, env=os.environ, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(proc.stdout[-4000:])
    status[ds] = "ok" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    print(f"{ds}: {status[ds]} in {(time.time() - t0) / 60:.1f} min", flush=True)
print(status)

In [ ]:
import pandas as pd
frames = []
for ds in DATASETS:
    suffix = "" if not MODELS else "_" + "_".join(w.strip().lower().replace(" ", "") for w in MODELS.split(",") if w.strip())
    path = f"{WORK}/results/{ds}/metrics_{MODE}{suffix}.csv"
    if os.path.exists(path):
        frames.append(pd.read_csv(path, index_col=0).assign(Dataset=ds))
summary = pd.concat(frames).reset_index().set_index(["Dataset", "Model"]) if frames else None
summary.round(4) if summary is not None else print("no results")